#### `Prebuild Middleware`

In [5]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

model = ChatGroq(model="openai/gpt-oss-120b")

#### `Summarization` middleware

In [6]:
CHAT = [
    "My order id is 4431",
    "It was a Nikon camera",
    "The box came open",
    "The lens cap is missing",
    "I want a replacement, not a refund"
]

agent = create_agent(
    system_prompt="""
        You are a helpful assistant, reply in one short line
    """,
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 6),
            keep=("messages", 2)
        )
    ]
)

In [8]:
config = { "configurable": {"thread_id": "user_2"}}
for message in CHAT:
    res = agent.invoke(
            { "messages": [ {"role": "user", "content": message},]},
            config=config
        )

    print(f"Messages in history: {len(agent.get_state(config).values["messages"])}")

Messages in history: 4
Messages in history: 6
Messages in history: 4
Messages in history: 6
Messages in history: 4


In [10]:
# agent.get_state(config).values["messages"]

In [11]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the product"}]},
    config=config
)

In [12]:
res["messages"][-1].content

'The product you ordered is a Nikon camera.'

#### `PIIMiddleware`

In [31]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="redact", apply_to_input=True)
    ],
)

In [34]:
query = "Hello, I recieved email from pujan123@gmail.com for the payment of credit card number 4242 4242 4242 4242"
res = agent.invoke({"messages": [{"role": "user", "content": query}]})

In [35]:
res["messages"][0].content

'Hello, I recieved email from [REDACTED_EMAIL] for the payment of credit card number [REDACTED_CREDIT_CARD]'